# Session 8 — Monge → Kantorovich

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

S1 *showed* optimal transport on a small example. S8 puts it on a formal footing.
We meet **Monge's** original deterministic formulation, see where it **breaks** (mass
splitting), build **Kantorovich's** linear-program relaxation, and end with the
**Birkhoff–von Neumann theorem** linking discrete OT to the **assignment problem**.

This is the LP that we will solve in S9 (Wasserstein distances), dualise in S10
(Sinkhorn), and **quantize** in S13–S14.

## 0. What you'll be able to do

- State the **Monge** and **Kantorovich** optimal-transport problems precisely.
- Construct a concrete example where **no deterministic Monge map exists** (mass must split) — and watch Kantorovich solve it anyway.
- Set up the discrete OT **linear program** with marginal constraints and read the resulting transport plan.
- Recognise the **transportation polytope** and the special **Birkhoff polytope** of doubly-stochastic matrices.
- See why equal-mass discrete OT reduces to the **assignment problem** (Birkhoff–von Neumann).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.transport.discrete import (
    squared_euclidean_cost,
    discrete_ot_plan,
    transport_cost,
    is_transportation_polytope_member,
    is_doubly_stochastic,
)

np.random.seed(42)
viz.use_course_style()

## 1. The transport problem, made precise

We have two histograms: a **source** $\mu = \sum_i a_i\,\delta_{x_i}$ on positions
$\{x_i\}_{i=1}^n$ with masses $a \in \Delta^n$, and a **target**
$\nu = \sum_j b_j\,\delta_{y_j}$ on positions $\{y_j\}_{j=1}^m$ with masses
$b \in \Delta^m$. We are given a **ground cost** $c(x, y) \ge 0$ — for instance the
squared distance — and we ask:

> *What is the cheapest way to reshape $\mu$ into $\nu$?*

Two answers exist. Monge (1781) gave the *deterministic* one. A century and a half later,
Kantorovich (1942) gave the **relaxation** that always works.

## 2. The Monge formulation

**Monge's problem.** Find a **transport map** $T : X \to Y$ that pushes every source atom
onto target atoms while respecting the target marginal, and minimise the total cost:

$$
\boxed{\;\min_{T :\, T_{\#}\mu = \nu}\ \int c(x, T(x))\,\mathrm{d}\mu(x)\;}
$$

In words: $T$ is a *function* — each source atom must go to one (and only one)
destination — and the **push-forward condition** $T_{\#}\mu = \nu$ says that the mass
arriving at any target $y$ must equal the mass of all sources sent to $y$.

This is **clean** when it works. But it can be **infeasible**. Watch.

## 3. Where Monge breaks — the mass-splitting obstruction

Take two source atoms at positions $0$ and $4$ with masses $1/2$ each. Take three target
atoms at positions $1$, $2$, $3$ with masses $1/3$ each. Total mass on each side equals
$1$ ✓. But — crucially — **a function $T$ assigns each of the 2 source atoms to a single
target**, so it can only deposit mass on at most 2 of the 3 target positions. **One
target will always be empty.** Monge is *infeasible*.

In [ ]:
x_source = np.array([0.0, 4.0])
x_target = np.array([1.0, 2.0, 3.0])
a = np.array([0.5, 0.5])
b = np.array([1/3, 1/3, 1/3])

viz.plot_distributions(a, b, source_positions=x_source, target_positions=x_target)
plt.show()

# Attempt a "naive Monge map": send each source to its nearest target.
nearest = np.array([np.argmin(np.abs(x_target - xs)) for xs in x_source])
delivered = np.zeros_like(b)
for src_idx, tgt_idx in enumerate(nearest):
    delivered[tgt_idx] += a[src_idx]
print(f"Source assigned to nearest target -> indices {nearest.tolist()}")
print(f"Delivered to each target:   {np.round(delivered, 3).tolist()}")
print(f"Required by each target:    {np.round(b, 3).tolist()}")
print(f"Push-forward satisfied?     {np.allclose(delivered, b)}")

**Read the figure and the output.** The "nearest-target" Monge attempt sends source 0
(at $x=0$) to target 0 (at $x=1$) and source 4 (at $x=4$) to target 2 (at $x=3$),
delivering $0.5$ to each of those targets — leaving target 1 (at $x=2$) with **zero**
mass instead of the required $1/3$. The push-forward condition fails *for any* Monge
map: with 2 source atoms and 3 target atoms requiring distinct masses, no deterministic
function can do the job. **Monge is infeasible here.**

## 4. The Kantorovich relaxation

Kantorovich's idea was to allow **fractional, non-deterministic** transport: rather than
a function $T$, work with a **joint distribution** $P$ on $X \times Y$ — a *coupling* of
$\mu$ and $\nu$. Now a single source atom can split its mass across several targets.

**The Kantorovich problem (discrete form).** Let $C \in \mathbb{R}_+^{n\times m}$ be the
cost matrix $C_{ij} = c(x_i, y_j)$. Solve the **linear program**

$$
\boxed{\;\min_{P \in \mathbb{R}_+^{n\times m}}\
   \langle C, P \rangle = \sum_{i, j} P_{ij}\, C_{ij}
   \quad \text{subject to} \quad
   P\,\mathbf{1} = a, \;\; P^\top \mathbf{1} = b. \;}
$$

The feasible set $T(a, b) = \{P \ge 0 : P\,\mathbf{1} = a,\ P^\top\,\mathbf{1} = b\}$ is
the **transportation polytope**: a bounded convex set defined by linear marginal
constraints. The LP optimum is attained at a *vertex* of this polytope.

Two crucial facts:
1. $T(a, b)$ is **never empty** for non-negative $a, b$ with $\sum a = \sum b$ — the
   *independent coupling* $P = a\, b^\top$ already lives in it.
2. Therefore Kantorovich is **always feasible**, even when Monge is not.

We let POT solve our 2 → 3 example.

In [ ]:
C = squared_euclidean_cost(x_source, x_target)
print("cost matrix C =")
print(C)

plan = discrete_ot_plan(a, b, C)
print(f"\noptimal plan P =")
print(np.round(plan, 6))

print(f"\nrow sums (should match a):    {np.round(plan.sum(axis=1), 6).tolist()}  ==  {a.tolist()}")
print(f"col sums (should match b):    {np.round(plan.sum(axis=0), 6).tolist()}  ==  {np.round(b, 6).tolist()}")
print(f"plan in T(a, b)?              {is_transportation_polytope_member(plan, a, b)}")
print(f"\ntotal transport cost <C, P> = {transport_cost(plan, C):.4f}")
print(f"sanity-check via ot.emd2     = {ot.emd2(a, b, C):.4f}")

**Read the output.** The optimal plan **splits both source atoms**:

- source $0$ ($x=0$, mass $1/2$) sends $1/3$ to target $0$ (closest, $x=1$) and $1/6$ to target $1$ ($x=2$);
- source $1$ ($x=4$, mass $1/2$) sends $1/3$ to target $2$ (closest, $x=3$) and $1/6$ to target $1$ ($x=2$).

Each row sums to the source mass; each column sums to the target mass; **the
push-forward holds exactly**. Kantorovich solves what Monge could not.

The total cost is the **2-Wasserstein squared distance** between $\mu$ and $\nu$ — but
the metric itself is the topic of S9. Today we focus on the LP and its structure.

In [ ]:
viz.plot_cost_matrix(C)
plt.show()

viz.plot_transport_plan(plan, title="Optimal plan $P^\\star$  (rows = source, cols = target)")
plt.show()

viz.plot_transport_arrows(
    a, b, plan,
    source_positions=x_source, target_positions=x_target,
    title="Flow view — source atoms split their mass (Monge impossible, Kantorovich solves)",
)
plt.show()

**Read the three figures.**

- **Cost matrix.** Dark bands sit close to the matching diagonal; expensive long-range moves are bright.
- **Plan heatmap.** Five non-zero cells in a 2×3 grid: each source row contributes mass to two distinct targets. *That mass splitting is exactly what no Monge map could express.*
- **Flow view.** Each green line carries mass from a source (top) to a target (bottom), its width proportional to the amount transported. You can literally see the two source piles "fanning out" to fill the three target piles.

## 5. The Birkhoff polytope and the assignment problem

Specialise to **uniform, equal-size** marginals: $a = b = \mathbf{1}_n / n$. Then
$T(a, b)$ becomes (up to a scaling by $n$) the **Birkhoff polytope**

$$
B_n = \{M \in \mathbb{R}_+^{n\times n} :\ M\mathbf{1} = \mathbf{1},\ M^\top \mathbf{1} = \mathbf{1}\}
$$

— the set of **doubly stochastic** $n\times n$ matrices. The fundamental structural
result is:

> **Birkhoff–von Neumann (1946).** The extreme points of $B_n$ are exactly the $n!$
> **permutation matrices**. Every doubly stochastic matrix is a *convex combination* of
> permutations.

So when Kantorovich's LP runs on uniform equal-size data, an LP vertex solution is
necessarily a permutation matrix — i.e. a **bijection** between source and target
atoms. The problem collapses to the **assignment problem**

$$
\min_{\sigma \in \mathcal{S}_n} \sum_{i=1}^n C_{i, \sigma(i)},
$$

solvable in $\mathcal{O}(n^3)$ by the Hungarian algorithm. *Monge was right after all
in this special case.*

In [ ]:
# Uniform equal-size example: source at [0, 1, 2], target at [3, 1, 5], mass 1/3 each.
x_src_uniform = np.array([0.0, 1.0, 2.0])
x_tgt_uniform = np.array([3.0, 1.0, 5.0])
a_unif = np.full(3, 1/3)
b_unif = np.full(3, 1/3)

C_unif = squared_euclidean_cost(x_src_uniform, x_tgt_uniform)
plan_unif = discrete_ot_plan(a_unif, b_unif, C_unif)
print("optimal plan (uniform, equal-size) =")
print(np.round(plan_unif, 3))

# Multiply by n=3: should be a 0/1 permutation matrix.
perm_matrix = 3 * plan_unif
print(f"\n3 * plan  (expected: a permutation matrix of 0s and 1s) =")
print(np.round(perm_matrix, 3))
print(f"\nIs 3*plan doubly stochastic?  {is_doubly_stochastic(perm_matrix)}")
print(f"Is each entry in {{0, 1}}?     {np.all(np.isin(np.round(perm_matrix), [0, 1]))}")

viz.plot_transport_arrows(
    a_unif, b_unif, plan_unif,
    source_positions=x_src_uniform, target_positions=x_tgt_uniform,
    title="Uniform equal-size OT — the optimal plan is a permutation (assignment problem)",
)
plt.show()

**Read the output and the flow view.** The optimal plan is *concentrated on 3 cells out of
9*, each of value $1/3$ — a permutation scaled by $1/3$. In the flow view, every source
atom sends *all* its mass to exactly one target. The matching follows the **sorted rank**
(a 1-D OT theorem we will revisit in S9): source rank-$k$ goes to target rank-$k$ once both
are sorted by position.

### Birkhoff decomposition — every doubly stochastic matrix is a mixture of permutations

We illustrate the structural side of Birkhoff–von Neumann on an explicit *interior*
doubly stochastic matrix.

In [ ]:
identity = np.eye(3)
cycle = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]], dtype=float)  # the 3-cycle (1 -> 2 -> 3 -> 1)

interior = 0.5 * identity + 0.5 * cycle
print("interior doubly stochastic matrix =")
print(interior)
print(f"\nrow sums = {interior.sum(axis=1).tolist()}")
print(f"col sums = {interior.sum(axis=0).tolist()}")
print(f"doubly stochastic? {is_doubly_stochastic(interior)}")

print("\nExplicit Birkhoff decomposition:  interior  =  0.5 * I  +  0.5 * cycle")
print(f"reconstruction matches?  {np.allclose(interior, 0.5 * identity + 0.5 * cycle)}")

# Visualise the three matrices side by side.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, M, name in zip(axes, [identity, cycle, interior],
                       ["permutation $P_1 = I$", "permutation $P_2$ (3-cycle)",
                        r"$\frac{1}{2}P_1 + \frac{1}{2}P_2$"]):
    im = ax.imshow(M, cmap=viz.CMAP_PLAN, vmin=0, vmax=1, origin="lower")
    ax.set_title(name, pad=10)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle("Birkhoff–von Neumann theorem: $B_n = \\mathrm{conv}\\{permutations\\}$",
             fontsize=14, fontweight="bold", y=1.04)
plt.tight_layout()
plt.show()

**Read the three matrices.** The leftmost and middle matrices are the two permutations
$I$ and a 3-cycle. Their convex combination $\tfrac{1}{2}P_1 + \tfrac{1}{2}P_2$
(rightmost) is doubly stochastic but **not a permutation** — its entries are $0$ and
$1/2$ rather than $0$ and $1$. *Every* element of $B_n$ has such a decomposition;
finding one is exactly what algorithms like Birkhoff's iterative subtraction do.

This polytope structure is the **geometric reason** the Kantorovich LP is so tractable:
its optima live on the vertices, and the vertices are simple combinatorial objects.

## 6. Dictionary update — the transport row begins

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ on positions $\{x_i\}$ | density matrix $\rho$ |
| marginal | partial trace |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ |
| Fisher–Rao metric | Bures / quantum Fisher metric |
| **transport plan / coupling $P \in T(a, b)$** | **quantum coupling**: bipartite $\rho_{AB}$ with $\mathrm{tr}_B\,\rho_{AB} = \rho_A$, $\mathrm{tr}_A\,\rho_{AB} = \rho_B$  *(S13)* |
| **Kantorovich LP** $\min \langle C, P\rangle$ | **quantum OT SDP** $\min \mathrm{tr}(C\,\rho_{AB})$  *(S13)* |
| **transportation polytope** $T(a, b)$ | **set of quantum couplings**  *(S13)* |
| **Birkhoff polytope / assignment problem** | (deterministic-coupling case, comes back in S15) |

The classical OT row now exists. Next session pulls the *cost* into a *distance*.

## 7. Your turn

1. **Build a Monge-fails example yourself.** Pick $n=3$ source atoms with masses
   $(0.2, 0.3, 0.5)$ and $m=3$ target atoms with masses $(0.4, 0.2, 0.4)$ on positions of
   your choice. Verify by hand that *no* deterministic $T$ pushes one onto the other; then
   solve the LP. How many non-zero entries does the optimal $P$ have?
2. **Counting LP vertices.** A vertex of the transportation polytope $T(a, b)$ — i.e. a
   basic feasible solution — has at most $n + m - 1$ non-zero entries (since each
   linearly independent marginal constraint kills one degree of freedom, and the row
   and column constraints together are not independent: their sums both equal $1$).
   Check this bound on the §4 optimum (where $n + m - 1 = 4$). Now construct a problem
   with a *degenerate* optimum (some $a_i$ or $b_j$ equal, or a tied cost) and see what
   happens to the count.
3. **Reverse-decompose.** Construct a $3\times3$ doubly stochastic matrix and write it as
   an explicit convex combination of two permutation matrices. (For a general matrix in
   $B_n$ the decomposition has at most $n^2 - 2n + 2$ permutations — Marcus–Newman.)

## Conclusions & references

- **Monge** asks for a deterministic map $T$; it can be **infeasible** when mass must split.
- **Kantorovich** relaxes to a coupling $P$ in the **transportation polytope** $T(a, b)$ — always non-empty, always solvable as a linear program.
- The **Birkhoff–von Neumann theorem** identifies the polytope's vertices as **permutation matrices**, linking discrete OT to the **assignment problem** in the equal-mass case.
- **Next (S9 — Wasserstein distances):** turn the LP cost into a *true metric* — the Wasserstein distance — and meet its closed form in 1-D, plus McCann's displacement interpolation we already met in S6.

**References:** G. Monge, "Mémoire sur la théorie des déblais et des remblais", *Hist.
Acad. R. Sci. Paris* (1781); L. V. Kantorovich, "On the translocation of masses",
*Dokl. Akad. Nauk SSSR* **37**, 199 (1942); G. Birkhoff, "Three observations on linear
algebra", *Univ. Nac. Tucumán Rev. A* **5**, 147 (1946); G. Peyré & M. Cuturi,
*Computational Optimal Transport* (NoW, 2019), chs.~2–3; C. Villani, *Topics in Optimal
Transportation* (AMS, 2003), ch.~1. Previous: `notebooks/02_InformationAndGeometry/03_quantum_information.ipynb`.